# Leslie Matrices — Part 2 Companion Notebook



This notebook accompanies **Leslie Matrices in Practice (Part 2)**.



## Learning goals

- Compare transient dynamics under different initial age structures

- Approximate elasticity of long-run growth to model parameters

- Explore how alternating environments change projections



## How to use this notebook

Run the notebook top to bottom, but treat each plot as an interpretation exercise: what changes because of the matrix, and what changes because of the initial condition or environment?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
L = np.array([
    [0.0, 1.1, 0.3, 0.0],
    [0.7, 0.0, 0.0, 0.0],
    [0.0, 0.8, 0.0, 0.0],
    [0.0, 0.0, 0.85, 0.0]
])

xA = np.array([40.0, 20.0, 10.0, 5.0])
xB = np.array([5.0, 10.0, 20.0, 40.0])

L

## Step 1 — Fix the demographic model



This matrix has four age classes, which gives a slightly richer life cycle than the Part 1 example.



Two initial populations are also defined:

- `xA` is concentrated in younger classes

- `xB` is concentrated in older classes



That setup lets us isolate a central idea: the same model can generate different short-run paths depending on the starting age structure.

In [ ]:
def project(L, x0, T=40):
    X = np.zeros((T + 1, len(x0)))
    X[0] = x0
    for t in range(T):
        X[t + 1] = L @ X[t]
    return X

XA = project(L, xA)
XB = project(L, xB)

In [ ]:
# Same matrix, different initial conditions: transient differences
plt.figure(figsize=(8, 4))
plt.plot(XA.sum(axis=1), label='Total pop (initial A)')
plt.plot(XB.sum(axis=1), label='Total pop (initial B)')
plt.title('Transient Dynamics Depend on Initial Age Structure')
plt.xlabel('Time step')
plt.ylabel('Total population')
plt.legend()
plt.tight_layout()
plt.show()

## Step 2 — Read transient dynamics carefully



The next plot compares total population size for the two initial states.



Even if two trajectories eventually reflect the same long-run growth mechanism, they can differ strongly in the first few periods. In real applications, those early differences are often the ones policymakers or ecologists care about most.

In [ ]:
def dominant_lambda(M):
    vals = np.linalg.eigvals(M)
    return np.max(vals.real)

def elasticity_fd(M, i, j, eps=1e-5):
    base = dominant_lambda(M)
    Mp = M.copy()
    Mp[i, j] += eps
    dlam = (dominant_lambda(Mp) - base) / eps
    return (M[i, j] / base) * dlam if M[i, j] != 0 else 0.0

base_lambda = dominant_lambda(L)
print(f'Base dominant eigenvalue: {base_lambda:.4f}')

## Step 3 — Measure parameter influence



Elasticity asks a policy-style question: if we slightly improve one fertility or survival entry, which change matters most for long-run growth?



The calculation below uses a finite-difference approximation. It is simple, transparent, and good enough for learning purposes, even though more exact formulas exist.

In [ ]:
# Elasticities for non-zero entries
elas = np.zeros_like(L)
for i in range(L.shape[0]):
    for j in range(L.shape[1]):
        if L[i, j] != 0:
            elas[i, j] = elasticity_fd(L, i, j)

print('Elasticity matrix (approx):')
print(np.round(elas, 4))

In [ ]:
# Time-varying environment: alternate between good and bad years
L_good = L.copy()
L_bad = L.copy()
L_bad[0, 1] *= 0.75  # lower fertility
L_bad[2, 1] *= 0.90  # lower survival transition

x = np.array([20.0, 20.0, 20.0, 20.0])
T = 50
traj = [x.copy()]
for t in range(T):
    Lt = L_good if t % 2 == 0 else L_bad
    x = Lt @ x
    traj.append(x.copy())
traj = np.array(traj)

plt.figure(figsize=(8, 4))
plt.plot(traj.sum(axis=1), label='Total population (alternating env)')
plt.title('Time-Varying Leslie Dynamics')
plt.xlabel('Time step')
plt.ylabel('Total population')
plt.legend()
plt.tight_layout()
plt.show()

## Step 4 — Explore changing environments



Now we leave the constant-matrix world. The simulation alternates between a good year and a bad year.



This illustrates an important modeling lesson: even when the underlying life cycle stays the same, environmental variability can reshape the trajectory in ways that a single fixed Leslie matrix cannot capture.

## Exercises and extensions



1. Replace alternating years with a random sequence of matrices and compare the resulting variability.

2. Compare elasticity rankings across two species profiles: one short-lived and one long-lived.

3. Add uncertainty by sampling fertility and survival values from probability distributions.

4. Write one paragraph explaining which intervention you would prioritize if this model described an endangered species.

5. Modify the initial condition and describe how your policy conclusion changes in the short run.